In [3]:
print("CSV RAG GEMINI")

CSV RAG GEMINI


In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
import os
load_dotenv()

True

In [5]:
GOOGLE_API_KEY=os.getenv("GOOGLE_API_KEY")

In [6]:
llm=ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.3
)

In [7]:
llm.invoke("What is Lang graph").content

'**LangGraph** is a library built on top of LangChain that allows you to build **stateful, multi-actor applications** with Large Language Models (LLMs) by representing your application\'s logic as a **graph**.\n\nThink of it as a powerful framework for orchestrating complex, multi-step LLM workflows, especially those involving agents that need to make decisions, use tools, and maintain a consistent state over time.\n\nHere\'s a breakdown of what that means:\n\n### The Core Problem It Solves\n\nTraditional LangChain "chains" are often linear or simple Directed Acyclic Graphs (DAGs). While powerful for many tasks, they can struggle with:\n\n1.  **State Management:** Maintaining a consistent, evolving state across multiple turns or steps, especially when an LLM needs to remember past interactions or tool outputs.\n2.  **Complex Control Flow:** Implementing conditional logic, loops, or the ability for an agent to "self-correct" or retry actions based on observations.\n3.  **Agentic Behavio

In [ ]:
from langchain.document_loaders.csv_loader import CSVLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
embed=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

C:\Users\HP\AppData\Local\Temp\ipykernel_5904\3102635589.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
loader_csv=CSVLoader("test.csv")
csv_doc=loader_csv.load()

In [ ]:
csv_doc

[Document(metadata={'source': 'test.csv', 'row': 0}, page_content='MovieID: 1\nTitle: Inception\nGenre: Sci-Fi\nRating: 8.8\nYear: 2010'),
 Document(metadata={'source': 'test.csv', 'row': 1}, page_content='MovieID: 2\nTitle: The Dark Knight\nGenre: Action\nRating: 9.0\nYear: 2008'),
 Document(metadata={'source': 'test.csv', 'row': 2}, page_content='MovieID: 3\nTitle: Interstellar\nGenre: Sci-Fi\nRating: 8.6\nYear: 2014'),
 Document(metadata={'source': 'test.csv', 'row': 3}, page_content='MovieID: 4\nTitle: Parasite\nGenre: Thriller\nRating: 8.6\nYear: 2019'),
 Document(metadata={'source': 'test.csv', 'row': 4}, page_content='MovieID: 5\nTitle: Avengers: Endgame\nGenre: Action\nRating: 8.4\nYear: 2019'),
 Document(metadata={'source': 'test.csv', 'row': 5}, page_content='MovieID: 6\nTitle: Titanic\nGenre: Romance\nRating: 7.8\nYear: 1997'),
 Document(metadata={'source': 'test.csv', 'row': 6}, page_content='MovieID: 7\nTitle: The Godfather\nGenre: Crime\nRating: 9.2\nYear: 1972'),
 Docume

In [ ]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=100,chunk_overlap=10)

In [ ]:
chunks=text_splitter.split_documents(csv_doc)

In [ ]:
for c in chunks:
    print(c)
    print('---')

page_content='MovieID: 1
Title: Inception
Genre: Sci-Fi
Rating: 8.8
Year: 2010' metadata={'source': 'test.csv', 'row': 0}
---
page_content='MovieID: 2
Title: The Dark Knight
Genre: Action
Rating: 9.0
Year: 2008' metadata={'source': 'test.csv', 'row': 1}
---
page_content='MovieID: 3
Title: Interstellar
Genre: Sci-Fi
Rating: 8.6
Year: 2014' metadata={'source': 'test.csv', 'row': 2}
---
page_content='MovieID: 4
Title: Parasite
Genre: Thriller
Rating: 8.6
Year: 2019' metadata={'source': 'test.csv', 'row': 3}
---
page_content='MovieID: 5
Title: Avengers: Endgame
Genre: Action
Rating: 8.4
Year: 2019' metadata={'source': 'test.csv', 'row': 4}
---
page_content='MovieID: 6
Title: Titanic
Genre: Romance
Rating: 7.8
Year: 1997' metadata={'source': 'test.csv', 'row': 5}
---
page_content='MovieID: 7
Title: The Godfather
Genre: Crime
Rating: 9.2
Year: 1972' metadata={'source': 'test.csv', 'row': 6}
---
page_content='MovieID: 8
Title: The Shawshank Redemption
Genre: Drama
Rating: 9.3
Year: 1994' meta

In [ ]:
vectorstores=FAISS.from_documents(chunks,embed)

In [ ]:
retriever=vectorstores.as_retriever(search_type='similarity',search_kwargs={'k':5})

In [ ]:
from langchain.chains import RetrievalQA
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
memory=ConversationBufferMemory()

In [ ]:
template="""You are a CSV bot. You will get context and always reply based on it.
Reply Should not be more than 50 words.
Here is input: {question}
Here is context: {context}"""
prompt=PromptTemplate(template=template,input_variables=['question','context'])

In [ ]:
csv_bot=RetrievalQA.from_chain_type(llm=llm,retriever=retriever,memory=memory,chain_type_kwargs={'prompt':prompt},chain_type='stuff')

NameError: name 'RetrievalQA' is not defined

In [ ]:
query="1997"

In [ ]:
similar_doc=retriever.get_relevant_documents(query)
print(similar_doc)

[Document(id='f08c017f-54b3-41c2-968e-981d111db89c', metadata={'source': 'test.csv', 'row': 5}, page_content='MovieID: 6\nTitle: Titanic\nGenre: Romance\nRating: 7.8\nYear: 1997'), Document(id='7d15129b-cdb0-4cf0-b3c2-27a96029882f', metadata={'source': 'test.csv', 'row': 6}, page_content='MovieID: 7\nTitle: The Godfather\nGenre: Crime\nRating: 9.2\nYear: 1972'), Document(id='1c2c6b1b-f643-48bf-8a48-13897c01351c', metadata={'source': 'test.csv', 'row': 4}, page_content='MovieID: 5\nTitle: Avengers: Endgame\nGenre: Action\nRating: 8.4\nYear: 2019'), Document(id='3f836af8-a76e-437c-9878-c8a5e42f69e9', metadata={'source': 'test.csv', 'row': 9}, page_content='MovieID: 10\nTitle: The Matrix\nGenre: Sci-Fi\nRating: 8.7\nYear: 1999'), Document(id='e901ccf7-678e-40d0-ae58-aa3127e1f787', metadata={'source': 'test.csv', 'row': 8}, page_content='MovieID: 9\nTitle: La La Land\nGenre: Romance\nRating: 8.0\nYear: 2016'), Document(id='a2b99710-8abd-4728-bd70-f6f2862da356', metadata={'source': 'test.cs

In [ ]:
response=csv_bot.invoke('Which movies rating is greater than 7')


In [ ]:
print('------CSV BOT-------')
print(response['result'])

------CSV BOT-------
The Matrix, Titanic, The Dark Knight, Interstellar, The Godfather, Inception, and The Shawshank Redemption all have a rating greater than 7.
